# 1 · A first look at PIEZO1

What is actually in a deposited structure, how to put it in a frame you can
reason about, and how to measure the membrane dome from the coordinates.

**You need the data first:**

```bash
python -m piezo1.io.fetch
```

Every cell below runs in a few seconds.

In [ ]:
from piezo1.config import STRUCTURE_DIR
from piezo1.core.structure import Structure

st = Structure.from_file(STRUCTURE_DIR / "8YEZ.cif")
print(st.n_atoms, "atoms |", st.n_residues, "residues |", st.chains, "chains")

## What is in the file, and what is the channel

A deposited entry is not only the protein you asked for. It carries lipids,
detergent, glycans, ions, water — and sometimes a whole other subunit. Six
PIEZO entries include three copies of **MDFIC**, a 21-residue auxiliary
protein whose residue numbers (226–247) sit *inside* PIEZO1's own range, so a
selection by residue number alone would silently mix the two.

`classify` sorts every atom, and the analyses use the channel protomers
whatever else is present.

In [ ]:
from piezo1.core.entities import classify

entities = classify(st)
for name, count in sorted(entities.counts().items(), key=lambda kv: -kv[1]):
    print(f"  {name:16s} {count:6d} atoms")
print("\nchannel protomer chains:", entities.protomer_chains)
print("auxiliary chains        :", entities.auxiliary_chains or "none")

## Put it in a frame you can reason about

Deposited entries sit wherever the depositor left them — up to 147 Å apart, and
some of them upside down. Every geometric number in this project (dome
curvature, pore profile, tag distance) is quoted relative to the three-fold
axis, so the structure has to be framed before any of it means anything.

`canonical_transform` finds the channel's own C3 axis, puts it on **+z**, and
puts the cytosolic side at **−z**.

In [ ]:
from piezo1.structure.frame import apply_frame, canonical_transform

transform = canonical_transform(st)
framed = apply_frame(st, transform)

print("mode              :", transform.mode)
print("C3 axis fit RMSD  :", round(transform.axis_rmsd, 3), "A")
print("note              :", transform.note)

# The cytosolic end must now be at negative z. Every distance quoted against
# the conduction axis depends on that sign being right.
#
# Select by residue NUMBER, not by position in the array. Taking "the last N
# rows" straddles chains and gave the wrong answer on 7WLU and 11ZC while
# still reporting a perfect C3 fit — a structure upside down in the viewport
# is obvious, one upside down inside a calculation is not.
mask = framed.mask_ca() & ~framed.hetero
seq, xyz = framed.res_seq[mask], framed.xyz[mask]
cterm = xyz[seq >= seq.max() - 40][:, 2].mean()
nterm = xyz[seq <= seq.min() + 40][:, 2].mean()
print(f"C-terminal (cytosolic) end: z = {cterm:+6.1f} A")
print(f"N-terminal end            : z = {nterm:+6.1f} A")
assert cterm < 0.0, "the cytosolic end must be at negative z"

## Measure the membrane dome

PIEZO1's blades bend the membrane into a dome, and the curvature of that dome
is the mechanism (Guo & MacKinnon 2017). The measurement fits a sphere to the
mid-membrane surface and reports the radius of curvature.

The regression case is curved mouse Piezo1 in a bilayer, **7WLT**: the code
should return about 9.7 nm against a published 10.2 nm (Haselwandter &
MacKinnon 2018).

In [ ]:
# ANALYSES is the shared registry the GUI and the command line both dispatch
# through, so a notebook using it cannot drift from what the application shows.
from piezo1.analysis.report import ANALYSES

curved = Structure.from_file(STRUCTURE_DIR / "7WLT.cif")
dome = ANALYSES["dome"](curved, "mouse")

for key, value in dome.items():
    if isinstance(value, float):
        print(f"  {key:24s} {value:8.2f}")
print(f"\n  reference: {dome['reference']}")

assert 9.0 < dome["radius_of_curvature_nm"] < 10.5, dome["radius_of_curvature_nm"]

## Curved against flat

The same measurement on the flattened state separates the two clearly. This is
the transition the whole project is about: tension flattens the dome, and the
blades lever the pore open.

In [ ]:
for pdb, species, state in (("7WLT", "mouse", "curved, bilayer"),
                            ("7WLU", "mouse", "flattened"),
                            ("11ZC", "mouse", "flat, native vesicle")):
    entry = Structure.from_file(STRUCTURE_DIR / f"{pdb}.cif")
    d = ANALYSES["dome"](entry, species)
    print(f"  {pdb}  {state:22s} R_c {d['radius_of_curvature_nm']:5.1f} nm   "
          f"depth {d['dome_depth_nm']:4.1f} nm")

## The trap that will bite you: residue numbering

Most PIEZO1 **mechanism** papers number by mouse Piezo1 (2,547 residues). Most
**disease** papers number by human PIEZO1 (2,521 residues). The offset between
them is **not constant** — it runs from 0 to +26 across twelve blocks and
passes through zero twice.

Never subtract a constant. Always go through the alignment.

In [ ]:
from piezo1.core.sequence import human_to_mouse

for human in (1718, 2456, 2496, 756):
    mouse = human_to_mouse(human)
    print(f"  human {human} -> mouse {mouse}   (offset {mouse - human:+d})")

offsets = {human_to_mouse(h) - h for h in range(600, 2500, 25)
           if human_to_mouse(h) is not None}
print("\ndistinct offsets across the chain:", sorted(offsets))
assert len(offsets) > 1, "a constant offset would mean the map is not being used"

## Where to go next

* `02_gating_motion` — the elastic network model, and the symmetry rule that
  says which motions can couple to membrane tension.
* `03_pore_to_current` — is the pore open, would water stay in it, and what
  current would flow.
* `04_variants_and_the_null` — the variant workflow, and the result that did
  not work.

`docs/NOTEBOOK.md` is the full API reference, with a table of the things that
will bite you.